In [ ]:
import requests
from urllib.parse import unquote

url = 'https://apis.data.go.kr/6260000/BusanCommercialHistoryService/getCommercialHistoryList'

service_key = 'mgnDSX5DP1DCeTiy5HhxKwhy0IkUeZDFUw%2Bib%2FftCc7dttQTy1aSMFcm8dtO%2BekyERqXPh84QRrxcL9OvFqDXg%3D%3D'

# 이미 Encoding된 키를 한 번 Decoding
service_key = unquote(service_key)

params = {
    'serviceKey': service_key,
    'pageNo': 1,
    'numOfRows': 10,
    'resultType': 'json'
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.text)

200
{"response":{"header":{"resultMsg":"NORMAL_CODE","resultCode":"00"},"body":{"totalCount":"336235","items":{"item":[{"trdstatenm":"폐업","bplcnm":"충렬손칼국수","majornm":"외식","minornm":"분식전문점","upjongnm":"기타 간이 음식점업","rdnwhladdr":" ","apvpermymd":"2005-09-08","dcbymd":"2006-11-15","geom":"POINT(129.107919799255 35.2033855635736)"},{"trdstatenm":"폐업","bplcnm":"아이엠푸드","majornm":"외식","minornm":"한식음식점","upjongnm":"한식 일반 음식점업","rdnwhladdr":"부산광역시 동래구 충렬대로348번길 23 (낙민동)","apvpermymd":"2005-09-09","dcbymd":"2022-12-07","geom":"POINT(129.095803493488 35.1966353543051)"},{"trdstatenm":"폐업","bplcnm":"김해뒷고기","majornm":"외식","minornm":"한식음식점","upjongnm":"한식 일반 음식점업","rdnwhladdr":" ","apvpermymd":"2005-09-13","dcbymd":"2010-11-24","geom":"POINT(129.105539275722 35.2007881729415)"},{"trdstatenm":"폐업","bplcnm":"거제횟집","majornm":"외식","minornm":"일식음식점","upjongnm":"일식 음식점업","rdnwhladdr":" ","apvpermymd":"2005-09-16","dcbymd":"2005-12-13","geom":"POINT(129.081723635797 35.2177911632851)"},{"trdstatenm":"폐업","b

In [ ]:
import pandas as pd

data = response.json()

items = data['response']['body']['items']['item']

df = pd.DataFrame(items)

df.head()

,trdstatenm,bplcnm,majornm,minornm,upjongnm,rdnwhladdr,apvpermymd,dcbymd,geom
0,폐업,충렬손칼국수,외식,분식전문점,기타 간이 음식점업,,2005-09-08,2006-11-15,POINT(129.107919799255 35.2033855635736)
1,폐업,아이엠푸드,외식,한식음식점,한식 일반 음식점업,부산광역시 동래구 충렬대로348번길 23 (낙민동),2005-09-09,2022-12-07,POINT(129.095803493488 35.1966353543051)
2,폐업,김해뒷고기,외식,한식음식점,한식 일반 음식점업,,2005-09-13,2010-11-24,POINT(129.105539275722 35.2007881729415)
3,폐업,거제횟집,외식,일식음식점,일식 음식점업,,2005-09-16,2005-12-13,POINT(129.081723635797 35.2177911632851)
4,폐업,미림레스토랑,외식,양식음식점,서양식 음식점업,,2005-09-23,2007-05-15,POINT(129.07972157347 35.2156525275316)


In [ ]:
df.to_csv(
    '/content/busan_facility.csv',
    index=False,
    encoding='utf-8-sig'
)

In [ ]:
data = response.json()

body = data['response']['body']

print('전체 데이터:', body['totalCount'])
print('현재 페이지:', body['pageNo'])
print('페이지당 데이터:', body['numOfRows'])

전체 데이터: 336235
현재 페이지: 85
페이지당 데이터: 1000


In [2]:
import requests
import pandas as pd
from urllib.parse import unquote
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# =========================
# 기본 설정
# =========================

# 네가 정상적으로 사용한 Endpoint
url = 'https://apis.data.go.kr/6260000/BusanCommercialHistoryService/getCommercialHistoryList'

# 공공데이터포털 일반 인증키
service_key = 'mgnDSX5DP1DCeTiy5HhxKwhy0IkUeZDFUw%2Bib%2FftCc7dttQTy1aSMFcm8dtO%2BekyERqXPh84QRrxcL9OvFqDXg%3D%3D'

# Encoding 키를 사용하고 있다면 한 번 Decoding
service_key = unquote(service_key)

num_of_rows = 1000
max_workers = 10


# =========================
# 1. 첫 번째 요청으로 전체 데이터 확인
# =========================

params = {
    'serviceKey': service_key,
    'pageNo': 1,
    'numOfRows': num_of_rows,
    'resultType': 'json'
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()
body = data['response']['body']

total_count = int(body['totalCount'])
total_pages = (total_count + num_of_rows - 1) // num_of_rows

print(f'전체 데이터: {total_count:,}건')
print(f'전체 페이지: {total_pages}페이지')


# =========================
# 2. 페이지 하나를 가져오는 함수
# =========================

def fetch_page(page):

    params = {
        'serviceKey': service_key,
        'pageNo': page,
        'numOfRows': num_of_rows,
        'resultType': 'json'
    }

    for attempt in range(3):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=30
            )

            response.raise_for_status()

            data = response.json()

            body = data['response']['body']
            items = body['items']['item']

            # 데이터가 1개일 경우에도 리스트로 변환
            if isinstance(items, dict):
                items = [items]

            return page, items

        except Exception as e:

            if attempt == 2:
                print(f'❌ {page}페이지 실패: {e}')
                return page, []

            time.sleep(1)


# =========================
# 3. 여러 페이지 동시에 수집
# =========================

all_items = []

# 1페이지 데이터는 이미 가져왔으므로 먼저 저장
items = body['items']['item']

if isinstance(items, dict):
    items = [items]

all_items.extend(items)

print(f'1/{total_pages} 페이지 완료')


# 2페이지부터 병렬 수집
with ThreadPoolExecutor(max_workers=max_workers) as executor:

    futures = [
        executor.submit(fetch_page, page)
        for page in range(2, total_pages + 1)
    ]

    completed = 1

    for future in as_completed(futures):

        page, items = future.result()

        all_items.extend(items)

        completed += 1

        print(
            f'\r수집 진행: {completed}/{total_pages} 페이지 '
            f'({len(all_items):,}/{total_count:,}건)',
            end=''
        )

print('\n\n수집 완료!')
print(f'실제 수집 데이터: {len(all_items):,}건')


# =========================
# 4. DataFrame 변환
# =========================

df = pd.DataFrame(all_items)

print(df.shape)
print("DataFrame 크기:", df.shape)
print("DataFrame 행 수:", len(df))

df = df.rename(columns={
    'trdstatenm': '영업상태',
    'bplcnm': '점포명',
    'majornm': '업종대분류',
    'minornm': '업종중분류',
    'upjongnm': '업종명',
    'rdnwhladdr': '도로명주소',
    'apvpermymd': '인허가일',
    'dcbymd': '폐업일',
    'geom': '좌표'
})


# =========================
# 5. CSV 저장
# =========================

csv_path = '/content/busan_store_history.csv'

df.to_csv(
    csv_path,
    index=False,
    encoding='utf-8-sig'
)

print('CSV 저장 완료!')
print(f'저장된 데이터: {len(df):,}건')

# =========================
# 6. CSV 다시 읽어서 검증
# =========================

check_df = pd.read_csv(csv_path)

print('\n===== CSV 검증 =====')
print(f'CSV 데이터: {len(check_df):,}건')
print(f'CSV 크기: {check_df.shape}')

print('\n[앞 5개]')
print(check_df.head())

print('\n[뒤 5개]')
print(check_df.tail())

전체 데이터: 336,235건
전체 페이지: 337페이지
1/337 페이지 완료
수집 진행: 337/337 페이지 (336,235/336,235건)

수집 완료!
실제 수집 데이터: 336,235건
(336235, 9)
DataFrame 크기: (336235, 9)
DataFrame 행 수: 336235
CSV 저장 완료!
저장된 데이터: 336,235건

===== CSV 검증 =====
CSV 데이터: 336,235건
CSV 크기: (336235, 9)

[앞 5개]
  영업상태     점포명 업종대분류  업종중분류         업종명                         도로명주소  \
0   폐업  충렬손칼국수    외식  분식전문점  기타 간이 음식점업                                 
1   폐업   아이엠푸드    외식  한식음식점  한식 일반 음식점업  부산광역시 동래구 충렬대로348번길 23 (낙민동)   
2   폐업   김해뒷고기    외식  한식음식점  한식 일반 음식점업                                 
3   폐업    거제횟집    외식  일식음식점     일식 음식점업                                 
4   폐업  미림레스토랑    외식  양식음식점    서양식 음식점업                                 

         인허가일         폐업일                                        좌표  
0  2005-09-08  2006-11-15  POINT(129.107919799255 35.2033855635736)  
1  2005-09-09  2022-12-07  POINT(129.095803493488 35.1966353543051)  
2  2005-09-13  2010-11-24  POINT(129.105539275722 35.2007881729415)  
3  2005-09-16  

In [3]:
from google.colab import files

files.download('/content/busan_store_history.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>